# 06 — Generate routing features

Produce annual nearest-infrastructure features for five non-PT destination types; cumulative 5/10/15/30-minute car accessibility; cumulative 5/10-minute pedestrian population, firm, public-transport, route-diversity, and reachable-cell features; firm-quarter accessibility; and the partitioned grid-quarter-Fachgruppe product. Public-transport stops are parent-station deduplicated and are handled only by pedestrian isochrones, not by the nearest-infrastructure stage.

In [ ]:
from __future__ import annotations

from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
import gc
import json
import itertools
import math
import os
import shutil
import numpy as np
import subprocess
import sys
import threading
import time

import duckdb
import geopandas as gpd
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import requests
from shapely.geometry import box, shape
from tqdm.auto import tqdm


def discover_project_dir() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "ANAL").is_dir() and (candidate / "OGD").is_dir():
            return candidate
    raise RuntimeError("Run this notebook from the project directory or a subdirectory containing ANAL/ and OGD/.")

PROJECT_DIR = discover_project_dir()
ANAL_DATA = PROJECT_DIR / "ANAL" / "data"
ROUTING_DATA = ANAL_DATA / "routing"
POI_DIR = ROUTING_DATA / "destinations"
ROUTING_DIR = PROJECT_DIR / "ANAL" / "routing"

ACTIVE_CELLS_PATH = ROUTING_DATA / "inputs" / "active_routing_cells_100m.parquet"
FIRMS_PATH = ANAL_DATA / "firms_assigned_100m.geoparquet"
PANEL_PATH = ANAL_DATA / "raster_quarter_panel_100m.parquet"
RASTER_PATH = ANAL_DATA / "raster_100m_styria.geoparquet"
FEATURE_ROOT = ROUTING_DATA / "features"
STATUS_DIR = ROUTING_DATA / "status"
ROUTING_STATUS_PATH = STATUS_DIR / "routing_feature_status.csv"

sys.path.insert(0, str(ROUTING_DIR))
from routing_utils import ACCESS_MINUTES, fachgruppe_access_columns, fachgruppe_ids, fachgruppe_stock_columns, main_access_columns

# Air-gapped setup: import/tag this exact image before running notebook 06. No pull is attempted.
VALHALLA_IMAGE = "valhalla-scripted:3.8.3"
VALHALLA_IMAGE_ID = "sha256:1e9f511e061eefde3ebab3b860517f06e14c31a24e88403a86365e64ce6adab4"
VALHALLA_PORT = 8002
VALHALLA_REPLICA_COUNT = 2
VALHALLA_PORTS = [VALHALLA_PORT + replica_index for replica_index in range(VALHALLA_REPLICA_COUNT)]
VALHALLA_URLS = [f"http://localhost:{port}" for port in VALHALLA_PORTS]
VALHALLA_URL = VALHALLA_URLS[0]
_valhalla_url_counter = itertools.count()
def valhalla_endpoint() -> str:
    return VALHALLA_URLS[next(_valhalla_url_counter) % len(VALHALLA_URLS)]
_accessibility_http_local = threading.local()
def accessibility_http_session() -> requests.Session:
    session = getattr(_accessibility_http_local, "session", None)
    if session is None:
        session = requests.Session()
        adapter = requests.adapters.HTTPAdapter(pool_connections=2, pool_maxsize=2)
        session.mount("http://", adapter)
        _accessibility_http_local.session = session
    return session
VALHALLA_CPUS = 8
VALHALLA_SERVER_THREADS = 10
WSL_EXE = r"C:\Windows\System32\wsl.exe"
WSL_DISTRO = "Ubuntu"
WSL_PROJECT_ROOT = "$HOME/gruendungsanalyse"
WSL_GRAPH_ROOT = f"{WSL_PROJECT_ROOT}/data/routing/valhalla_graphs"
START_YEAR, END_YEAR = 2015, 2025
FACHGRUPPE_IDS = fachgruppe_ids(PANEL_PATH)
if len(FACHGRUPPE_IDS) != 95:
    raise ValueError(f"Expected 95 Fachgruppen, found {len(FACHGRUPPE_IDS)}")

FEATURE_ROOT.mkdir(parents=True, exist_ok=True)
STATUS_DIR.mkdir(parents=True, exist_ok=True)

## Run configuration

The deterministic configuration defaults to all years, four sequential slices, one local pinned Valhalla service, conservative concurrency, automatic checkpoints, bounded retries, and cleanup after validation. Use `RUN_MODE = "smoke"` only for a local route check; `dry-run` prints the configuration without writing.

In [ ]:
RUN_MODE = "full"  # "dry-run", "smoke", or "full"
YEARS_TO_RUN = list(range(START_YEAR, END_YEAR + 1))
REGENERATE_OUTPUTS = True
AUTO_SLICE_COUNT = 4
RUN_NEAREST_INFRASTRUCTURE = True
RUN_ACCESSIBILITY = True
DESTINATION_TYPES = ["motorway_exit", "regional_centre", "urban_centre", "higher_education"]
FORBIDDEN_NEAREST_COLUMNS = {"tt_rail_station_min", "km_rail_station", "nearest_rail_station_id", "routing_status_rail_station"}
MAX_DESTINATIONS_PER_TYPE = None
VALHALLA_MAX_MATRIX_PAIRS = 2500
ORIGIN_CHUNK_SIZE = 24
DESTINATION_CHUNK_SIZE = 100
MAX_CONCURRENT_REQUESTS = 20
NEAREST_INFRA_EUCLIDEAN_PREFILTER_M = 10_000
NEAREST_INFRA_DESTINATION_STEP_SHARE = 0.10
NEAREST_INFRA_TYPE_WORKERS = 1
NEAREST_INFRA_ORIGIN_CHUNK_WORKERS = max(1, math.ceil(MAX_CONCURRENT_REQUESTS / NEAREST_INFRA_TYPE_WORKERS))
NEAREST_INFRA_MAX_IN_FLIGHT_ORIGIN_CHUNKS = NEAREST_INFRA_ORIGIN_CHUNK_WORKERS * 4
POPULATION_ACCESS_COSTING = "auto"
POPULATION_ACCESS_CONTOURS_MIN = [5, 10, 15, 30]
PEDESTRIAN_ACCESS_COSTING = "pedestrian"
PEDESTRIAN_ACCESS_CONTOURS_MIN = [5, 10]
POPULATION_ACCESS_POLYGONS = True
POPULATION_ACCESS_DENOISE = 0
POPULATION_ACCESS_GENERALIZE_M = 0
POPULATION_ACCESS_ORIGIN_WORKERS = 32
POPULATION_ACCESS_MAX_IN_FLIGHT_ORIGINS = 64
POPULATION_ACCESS_FLUSH_ORIGINS = 500
POPULATION_ACCESS_REQUEST_TIMEOUT_SECONDS = 600
POPULATION_ACCESS_REQUEST_RETRIES = 3
ACCESSIBILITY_SKIPPED_ORIGINS = {
    skipped_year: {
        "AT_CRS3035RES100mN2655400E4708200": "Valhalla 30-minute isochrone hangs deterministically in 2020 and 2021; remote cell excluded consistently for 2020-2025",
    }
    for skipped_year in range(2020, END_YEAR + 1)
}
ACCESSIBILITY_SKIPPED_ORIGINS_PATH = ROUTING_DATA / "status" / "accessibility_skipped_origins.csv"
MATRIX_REQUEST_TIMEOUT_SECONDS = 300
ORIGIN_SLICE_INDEX = 0
ORIGIN_SLICE_COUNT = AUTO_SLICE_COUNT
SLICE_ONLY = False
EXTERNAL_VALHALLA = False
VALHALLA_RESTART_ATTEMPTS = 3
MAX_WAIT_MINUTES = 30
MAX_ACTIVE_CELLS = None

## WSL / Docker Helpers

In [ ]:
def run_local(command: list[str], check: bool = True, capture_output: bool = True) -> subprocess.CompletedProcess:
    return subprocess.run(command, check=check, text=True, capture_output=capture_output)


def wsl_base_command() -> list[str]:
    if WSL_DISTRO:
        return [WSL_EXE, "-d", WSL_DISTRO, "--"]
    return [WSL_EXE, "--"]


def run_wsl(command: str, check: bool = True, capture_output: bool = True) -> subprocess.CompletedProcess:
    return run_local([*wsl_base_command(), "bash", "-lc", command], check=check, capture_output=capture_output)


def require_wsl_distribution() -> None:
    distro_list = run_local([WSL_EXE, "-l", "-q"], check=False)
    available_distros = [line.strip().replace("\x00", "") for line in distro_list.stdout.splitlines() if line.strip().replace("\x00", "")]
    if WSL_DISTRO and available_distros and WSL_DISTRO not in available_distros:
        raise RuntimeError(
            f"Configured WSL_DISTRO={WSL_DISTRO!r}, but PowerShell reports these WSL distros: {available_distros}. "
            "Set WSL_DISTRO to the exact name from `wsl -l -v`."
        )
    test = run_wsl("printf ok", check=False)
    if test.returncode != 0:
        raise RuntimeError(
            "Could not start the configured WSL distribution. Run `wsl -l -v` in PowerShell "
            "and set WSL_DISTRO in this notebook to the exact distro name.\n\n"
            f"stdout:\n{test.stdout}\n\nstderr:\n{test.stderr}"
        )


def quote_bash(value: str) -> str:
    return "'" + value.replace("'", "'\\''") + "'"


def quote_wsl_path(value: str) -> str:
    if value.startswith("$HOME/"):
        return "$HOME/" + quote_bash(value.removeprefix("$HOME/"))
    return quote_bash(value)


def wsl_graph_dir(year: int) -> str:
    return f"{WSL_GRAPH_ROOT}/{year}"


def wsl_manifest_path(year: int) -> str:
    return f"{wsl_graph_dir(year)}/build_manifest.json"


def container_name(year: int) -> str:
    return f"co2-valhalla-routing-{year}"


def container_names(year: int) -> list[str]:
    base_name = container_name(year)
    return [base_name, *[f"{base_name}-replica-{replica_index}" for replica_index in range(1, VALHALLA_REPLICA_COUNT)]]


def graph_manifest_exists(year: int) -> bool:
    return run_wsl(f"test -f {quote_wsl_path(wsl_manifest_path(year))}", check=False).returncode == 0


def start_valhalla_container(year: int) -> None:
    if not graph_manifest_exists(year):
        raise FileNotFoundError(f"Missing graph manifest in WSL: {wsl_manifest_path(year)}")
    graph_dir = wsl_graph_dir(year)
    commands = []
    for replica_index, (name, port) in enumerate(zip(container_names(year), VALHALLA_PORTS)):
        commands.extend([
            f"docker rm -f {quote_bash(name)} >/dev/null 2>&1 || true",
            "docker run -d "
            f"--name {quote_bash(name)} "
            f"--cpus {VALHALLA_CPUS} "
            f"-p {port}:8002 "
            "-e build_admins=False "
            "-e build_time_zones=False "
            "-e build_tar=False "
            "-e build_transit=False "
            "-e use_default_speeds_config=False "
            "-e update_existing_config=False "
            "-e serve_tiles=True "
            f"-e server_threads={VALHALLA_SERVER_THREADS} "
            f"-v {quote_wsl_path(graph_dir)}:/custom_files "
            f"{quote_bash(VALHALLA_IMAGE)}",
        ])
    run_wsl(" && ".join(commands))


def stop_valhalla_container(year: int) -> None:
    commands = [f"docker rm -f {quote_bash(name)} >/dev/null 2>&1 || true" for name in container_names(year)]
    run_wsl(" && ".join(commands), check=False)


def wait_until_valhalla_ready(year: int, max_wait_minutes: int = MAX_WAIT_MINUTES) -> dict:
    deadline = time.time() + max_wait_minutes * 60
    last_error = None
    while time.time() < deadline:
        try:
            summary = valhalla_test_route()
            print(f"Valhalla ready for {year}: {summary}")
            return summary
        except Exception as error:
            last_error = error
            time.sleep(10)
    logs = run_wsl(f"docker logs --tail 80 {quote_bash(container_name(year))}", check=False).stdout
    raise TimeoutError(f"Valhalla did not become ready for {year}. Last error: {last_error}\n\nContainer logs:\n{logs}")

## Routing Helpers

In [ ]:
def output_paths(year: int) -> dict:
    feature_dir = FEATURE_ROOT / str(year)
    feature_dir.mkdir(parents=True, exist_ok=True)
    slice_suffix = f".slice{ORIGIN_SLICE_INDEX}" if ORIGIN_SLICE_COUNT > 1 else ""
    return {
        "nearest": feature_dir / "nearest_infrastructure_100m.parquet",
        "nearest_partial": feature_dir / "nearest_infrastructure_100m.partial.parquet",
        "potentials": feature_dir / f"accessibility_potentials_100m{slice_suffix}.parquet",
        "potentials_parts": feature_dir / f".accessibility_parts{slice_suffix}",
        "pedestrian_accessibility": feature_dir / "pedestrian_accessibility_quarter_100m.parquet",
        "fachgruppe_accessibility": feature_dir / "fachgruppe_accessibility_quarter_100m.parquet",
        "firm_accessibility": feature_dir / "firm_accessibility_quarter_100m.parquet",
    }


def valhalla_test_route() -> dict:
    payload = {
        "locations": [
            {"lat": 47.0707, "lon": 15.4395},
            {"lat": 47.0580, "lon": 15.4630},
        ],
        "costing": "auto",
        "directions_options": {"units": "kilometers"},
    }
    response = requests.post(f"{valhalla_endpoint()}/route", json=payload, timeout=30)
    response.raise_for_status()
    summary = response.json()["trip"]["summary"]
    if summary.get("time", 0) <= 0:
        raise RuntimeError(f"Valhalla responded but returned an invalid route summary: {summary}")
    return summary


def route_matrix(origins: pd.DataFrame, destinations: gpd.GeoDataFrame, costing: str = "auto") -> list[list[dict]]:
    matrix_pairs = len(origins) * len(destinations)
    if matrix_pairs > VALHALLA_MAX_MATRIX_PAIRS:
        raise ValueError(
            f"Matrix request has {matrix_pairs:,} pairs "
            f"({len(origins):,} origins * {len(destinations):,} destinations), "
            f"above Valhalla limit {VALHALLA_MAX_MATRIX_PAIRS:,}. "
            "Lower ORIGIN_CHUNK_SIZE or DESTINATION_CHUNK_SIZE."
        )
    payload = {
        "sources": [
            {"lat": float(row.lat), "lon": float(row.lon)}
            for row in origins.itertuples(index=False)
        ],
        "targets": [
            {"lat": float(row.lat), "lon": float(row.lon)}
            for row in destinations.itertuples(index=False)
        ],
        "costing": costing,
    }
    last_error = None
    for attempt in range(1, 4):
        try:
            response = requests.post(f"{valhalla_endpoint()}/sources_to_targets", json=payload, timeout=MATRIX_REQUEST_TIMEOUT_SECONDS)
            break
        except requests.RequestException as error:
            last_error = error
            if attempt == 3:
                raise RuntimeError(f"Valhalla matrix request failed after retries: {error}") from error
            time.sleep(2 ** (attempt - 1))
    if last_error is not None and "response" not in locals():
        raise RuntimeError(f"Valhalla matrix request failed: {last_error}") from last_error
    if not response.ok:
        if response.status_code == 400 and "unconnected regions" in response.text.lower():
            return [[{"status": 1} for _ in range(len(destinations))] for _ in range(len(origins))]
        raise RuntimeError(
            f"Valhalla matrix request failed with HTTP {response.status_code}: {response.text[:1000]}"
        )
    data = response.json()
    if "sources_to_targets" not in data:
        raise RuntimeError(f"Unexpected Valhalla matrix response keys: {sorted(data.keys())}")
    return data["sources_to_targets"]


def route_matrix_chunk(
    origins: pd.DataFrame,
    destinations: gpd.GeoDataFrame,
    costing: str = "auto",
) -> tuple[pd.DataFrame, gpd.GeoDataFrame, list[list[dict]]]:
    return origins, destinations, route_matrix(origins, destinations, costing=costing)


def finite_number(value: object) -> bool:
    return isinstance(value, (int, float)) and math.isfinite(value)


def normalize_destinations(pois: gpd.GeoDataFrame, poi_type: str) -> gpd.GeoDataFrame:
    selected = pois[pois["poi_type"] == poi_type].copy()
    if selected.empty:
        return selected
    selected = selected.to_crs("EPSG:4326")
    selected["lon"] = selected.geometry.x
    selected["lat"] = selected.geometry.y
    if "poi_id" not in selected.columns:
        selected["poi_id"] = poi_type + "_" + selected.index.astype(str)
    if MAX_DESTINATIONS_PER_TYPE is not None:
        selected = selected.head(MAX_DESTINATIONS_PER_TYPE).copy()
    return selected.reset_index(drop=True)


def nearest_infrastructure_stage_size(total_destinations: int, step_share: float | None = None) -> int:
    share = NEAREST_INFRA_DESTINATION_STEP_SHARE if step_share is None else step_share
    return max(1, math.ceil(total_destinations * share))


def update_best_routes(
    best_by_origin: dict,
    origins: pd.DataFrame,
    destinations: gpd.GeoDataFrame,
    matrix: list[list[dict]],
    eligible_destination_indices_by_origin: dict | None = None,
) -> None:
    for origin_row, result_row in zip(origins.itertuples(index=False), matrix):
        current_best = best_by_origin.get(origin_row.grid_id)
        eligible_indices = None
        if eligible_destination_indices_by_origin is not None:
            eligible_indices = eligible_destination_indices_by_origin.get(origin_row.grid_id)
        for destination_index, result in enumerate(result_row):
            destination_global_index = int(destinations.index[destination_index])
            if eligible_indices is not None and destination_global_index not in eligible_indices:
                continue
            if result.get("status", 0) != 0:
                continue
            result_time = result.get("time")
            result_distance = result.get("distance")
            if not finite_number(result_time) or not finite_number(result_distance):
                continue
            current_best_time = math.inf if current_best is None else current_best["result"].get("time", math.inf)
            if result_time < current_best_time:
                current_best = {
                    "result": result,
                    "destination": destinations.iloc[destination_index],
                }
        best_by_origin[origin_row.grid_id] = current_best


def best_routes_to_records(best_by_origin: dict, poi_type: str, missing_status_by_origin: dict | None = None) -> pd.DataFrame:
    records = []
    for grid_id, best_payload in best_by_origin.items():
        if best_payload is None:
            missing_status = "unroutable" if missing_status_by_origin is None else missing_status_by_origin.get(grid_id, "unroutable")
            records.append({
                "grid_id": grid_id,
                f"tt_{poi_type}_min": pd.NA,
                f"km_{poi_type}": pd.NA,
                f"nearest_{poi_type}_id": pd.NA,
                f"routing_status_{poi_type}": missing_status,
            })
            continue
        best = best_payload["result"]
        destination = best_payload["destination"]
        records.append({
            "grid_id": grid_id,
            f"tt_{poi_type}_min": float(best.get("time")) / 60,
            f"km_{poi_type}": best.get("distance"),
            f"nearest_{poi_type}_id": destination.get("poi_id"),
            f"routing_status_{poi_type}": "ok",
        })
    columns = [
        "grid_id",
        f"tt_{poi_type}_min",
        f"km_{poi_type}",
        f"nearest_{poi_type}_id",
        f"routing_status_{poi_type}",
    ]
    return pd.DataFrame(records, columns=columns)


def nearest_for_origin_chunk(
    origins: pd.DataFrame,
    destinations: gpd.GeoDataFrame,
    destination_x: np.ndarray,
    destination_y: np.ndarray,
    poi_type: str,
    costing: str = "auto",
    euclidean_prefilter_m: int = NEAREST_INFRA_EUCLIDEAN_PREFILTER_M,
    destination_step_share: float | None = None,
) -> tuple[pd.DataFrame, int]:
    best_by_origin = {grid_id: None for grid_id in origins["grid_id"]}
    total_destinations = len(destinations)
    stage_size = nearest_infrastructure_stage_size(total_destinations, destination_step_share)
    radius_sq = euclidean_prefilter_m ** 2
    completed_requests = 0

    origin_grid_ids = origins["grid_id"].tolist()
    origin_x = origins["centroid_x_3035"].to_numpy(dtype=float)
    origin_y = origins["centroid_y_3035"].to_numpy(dtype=float)
    distance_sq = (origin_x[:, None] - destination_x[None, :]) ** 2 + (origin_y[:, None] - destination_y[None, :]) ** 2
    ordered_destination_indices = np.argsort(distance_sq, axis=1)
    prefilter_counts = (distance_sq <= radius_sq).sum(axis=1).astype(int)
    initial_limits = np.maximum(stage_size, prefilter_counts)
    missing_status_by_origin = None
    current_limits = np.clip(initial_limits, 1, total_destinations)
    previous_limits = np.zeros(len(origins), dtype=int)
    unresolved_positions = np.arange(len(origins), dtype=int)

    while len(unresolved_positions) > 0:
        eligible_by_origin = {}
        pending_positions = []
        requested_destination_indices = set()
        for row_position in unresolved_positions:
            start = int(previous_limits[row_position])
            stop = int(current_limits[row_position])
            if start >= stop:
                continue
            candidate_indices = {
                int(index)
                for index in ordered_destination_indices[row_position, start:stop]
            }
            if not candidate_indices:
                continue
            pending_positions.append(int(row_position))
            eligible_by_origin[origin_grid_ids[row_position]] = candidate_indices
            requested_destination_indices.update(candidate_indices)

        if not requested_destination_indices:
            break

        pending_origins = origins.iloc[pending_positions].copy()
        destination_indices = sorted(requested_destination_indices)
        for destination_start in range(0, len(destination_indices), DESTINATION_CHUNK_SIZE):
            chunk_indices = destination_indices[destination_start:destination_start + DESTINATION_CHUNK_SIZE]
            destination_chunk = destinations.iloc[chunk_indices].copy()
            matrix = route_matrix(pending_origins, destination_chunk, costing=costing)
            update_best_routes(
                best_by_origin,
                pending_origins,
                destination_chunk,
                matrix,
                eligible_destination_indices_by_origin=eligible_by_origin,
            )
            completed_requests += 1

        previous_limits[pending_positions] = current_limits[pending_positions]
        next_unresolved_positions = [
            int(row_position)
            for row_position in unresolved_positions
            if best_by_origin[origin_grid_ids[row_position]] is None and current_limits[row_position] < total_destinations
        ]
        if not next_unresolved_positions:
            break
        current_limits[next_unresolved_positions] = np.minimum(
            total_destinations,
            current_limits[next_unresolved_positions] + stage_size,
        )
        unresolved_positions = np.array(next_unresolved_positions, dtype=int)

    return best_routes_to_records(best_by_origin, poi_type, missing_status_by_origin), completed_requests


def nearest_for_type(
    active_cells: pd.DataFrame,
    destinations: gpd.GeoDataFrame,
    poi_type: str,
    progress_position: int | None = None,
    costing: str = "auto",
    euclidean_prefilter_m: int = NEAREST_INFRA_EUCLIDEAN_PREFILTER_M,
    destination_step_share: float | None = None,
) -> pd.DataFrame:
    destinations_3035 = destinations.to_crs("EPSG:3035")
    destination_x = destinations_3035.geometry.x.to_numpy(dtype=float)
    destination_y = destinations_3035.geometry.y.to_numpy(dtype=float)
    chunk_starts = list(range(0, len(active_cells), ORIGIN_CHUNK_SIZE))
    completed_requests = 0
    chunk_results = {}
    progress_kwargs = {
        "total": len(active_cells),
        "desc": f"Nearest infrastructure: {poi_type}",
        "unit": "origin",
        "leave": True,
    }
    if progress_position is not None:
        progress_kwargs["position"] = progress_position
    progress_bar = tqdm(**progress_kwargs)

    def submit_next(executor, next_chunk_index: int, futures: dict) -> int:
        if next_chunk_index >= len(chunk_starts):
            return next_chunk_index
        chunk_start = chunk_starts[next_chunk_index]
        origins = active_cells.iloc[chunk_start:chunk_start + ORIGIN_CHUNK_SIZE].copy()
        future = executor.submit(
            nearest_for_origin_chunk,
            origins,
            destinations,
            destination_x,
            destination_y,
            poi_type,
            costing,
            euclidean_prefilter_m,
            destination_step_share,
        )
        futures[future] = chunk_start
        return next_chunk_index + 1

    try:
        with ThreadPoolExecutor(max_workers=NEAREST_INFRA_ORIGIN_CHUNK_WORKERS) as executor:
            futures = {}
            next_chunk_index = 0
            while next_chunk_index < len(chunk_starts) and len(futures) < NEAREST_INFRA_MAX_IN_FLIGHT_ORIGIN_CHUNKS:
                next_chunk_index = submit_next(executor, next_chunk_index, futures)

            while futures:
                for completed in as_completed(list(futures)):
                    chunk_start = futures.pop(completed)
                    nearest_chunk, request_count = completed.result()
                    chunk_results[chunk_start] = nearest_chunk
                    completed_requests += request_count
                    progress_bar.update(len(nearest_chunk))
                    progress_bar.set_postfix_str(f"{completed_requests:,} matrix requests")
                    while next_chunk_index < len(chunk_starts) and len(futures) < NEAREST_INFRA_MAX_IN_FLIGHT_ORIGIN_CHUNKS:
                        next_chunk_index = submit_next(executor, next_chunk_index, futures)
                    break
    finally:
        progress_bar.close()

    if not chunk_results:
        return best_routes_to_records({}, poi_type)
    return pd.concat([chunk_results[chunk_start] for chunk_start in chunk_starts], ignore_index=True)


def merge_nearest_results(base_output: pd.DataFrame, destination_tables: dict, nearest_results_by_type: dict) -> pd.DataFrame:
    output = base_output.copy()
    for poi_type in DESTINATION_TYPES:
        destinations = destination_tables[poi_type]
        if destinations.empty:
            output[f"tt_{poi_type}_min"] = pd.NA
            output[f"km_{poi_type}"] = pd.NA
            output[f"nearest_{poi_type}_id"] = pd.NA
            output[f"routing_status_{poi_type}"] = "missing_destinations"
            continue
        nearest = nearest_results_by_type.get(poi_type)
        if nearest is None:
            continue
        output = output.merge(nearest, on="grid_id", how="left")
    return output


def generate_nearest_infrastructure(year: int) -> Path:
    paths = output_paths(year)
    poi_path = POI_DIR / f"austria-{year}-pois.geoparquet"
    if not poi_path.exists():
        raise FileNotFoundError(f"Missing yearly POI file: {poi_path}")

    active_cells = pd.read_parquet(ACTIVE_CELLS_PATH)
    if MAX_ACTIVE_CELLS is not None:
        active_cells = active_cells.head(MAX_ACTIVE_CELLS).copy()

    pois = gpd.read_parquet(poi_path)
    completed_types = set()
    if paths["nearest_partial"].exists():
        checkpoint = pd.read_parquet(paths["nearest_partial"])
        legacy_pt_columns = {"tt_pt_stop_min", "km_pt_stop", "nearest_pt_stop_id", "routing_status_pt_stop", "has_pt_stop_5min_walk", "pt_departures_5min_walk"}
        if len(checkpoint) == len(active_cells) and checkpoint["grid_id"].equals(active_cells["grid_id"]) and not legacy_pt_columns.intersection(checkpoint.columns) and not FORBIDDEN_NEAREST_COLUMNS.intersection(checkpoint.columns):
            base_output = checkpoint.drop(columns=["year", "created_at"], errors="ignore").copy()
            completed_types = {poi_type for poi_type in DESTINATION_TYPES if f"routing_status_{poi_type}" in checkpoint.columns}
            print(f"Resuming {year}: completed destination types={sorted(completed_types)}")
        else:
            paths["nearest_partial"].unlink()
    if "base_output" not in locals() or len(base_output) != len(active_cells):
        base_output = active_cells[["grid_id"]].copy()
    available_types = sorted(pois["poi_type"].dropna().unique()) if "poi_type" in pois.columns else []
    print(f"{year}: {len(active_cells):,} origins, POI types: {available_types}")

    destination_tables = {poi_type: normalize_destinations(pois, poi_type) for poi_type in DESTINATION_TYPES}
    nearest_results_by_type = {}
    non_empty_types = []
    for poi_type in DESTINATION_TYPES:
        destinations = destination_tables[poi_type]
        if poi_type in completed_types:
            continue
        if destinations.empty:
            tqdm.write(f"Skip {poi_type}: no destinations in {poi_path.name}")
            continue
        non_empty_types.append(poi_type)

    if non_empty_types:
        type_positions = {poi_type: index for index, poi_type in enumerate(non_empty_types)}
        with ThreadPoolExecutor(max_workers=min(NEAREST_INFRA_TYPE_WORKERS, len(non_empty_types))) as executor:
            futures = {
                executor.submit(
                    nearest_for_type,
                    active_cells,
                    destination_tables[poi_type],
                    poi_type,
                    type_positions[poi_type],
                    "auto",
                    NEAREST_INFRA_EUCLIDEAN_PREFILTER_M,
                ): poi_type
                for poi_type in non_empty_types
            }
            for completed in as_completed(futures):
                poi_type = futures[completed]
                nearest_results_by_type[poi_type] = completed.result()
                checkpoint = merge_nearest_results(base_output, destination_tables, nearest_results_by_type)
                checkpoint["year"] = year
                checkpoint["created_at"] = datetime.now().isoformat(timespec="seconds")
                checkpoint.to_parquet(paths["nearest_partial"], index=False)
                tqdm.write(f"Checkpointed nearest infrastructure after {poi_type} to {paths['nearest_partial']}")

    output = merge_nearest_results(base_output, destination_tables, nearest_results_by_type)
    output["year"] = year
    output["created_at"] = datetime.now().isoformat(timespec="seconds")
    nearest_tmp_path = paths["nearest"].with_name(paths["nearest"].name + ".tmp")
    output.to_parquet(nearest_tmp_path, index=False)
    nearest_tmp_path.replace(paths["nearest"])
    if paths["nearest_partial"].exists():
        paths["nearest_partial"].unlink()
    print(f"Wrote {len(output):,} rows to {paths['nearest']}")
    return paths["nearest"]


def load_active_cells_for_run() -> pd.DataFrame:
    active_cells = pd.read_parquet(ACTIVE_CELLS_PATH)
    if MAX_ACTIVE_CELLS is not None:
        active_cells = active_cells.head(MAX_ACTIVE_CELLS).copy()
    if ORIGIN_SLICE_COUNT > 1:
        if ORIGIN_SLICE_INDEX >= ORIGIN_SLICE_COUNT:
            raise ValueError(f"ROUTING_ORIGIN_SLICE_INDEX={ORIGIN_SLICE_INDEX} must be less than count={ORIGIN_SLICE_COUNT}")
        start = (len(active_cells) * ORIGIN_SLICE_INDEX) // ORIGIN_SLICE_COUNT
        stop = (len(active_cells) * (ORIGIN_SLICE_INDEX + 1)) // ORIGIN_SLICE_COUNT
        active_cells = active_cells.iloc[start:stop].copy()
    return active_cells.reset_index(drop=True)

In [ ]:
def year_quarters(year: int) -> pd.PeriodIndex:
    return pd.period_range(f"{year}Q1", f"{year}Q4", freq="Q")


def accessibility_minutes() -> list[int]:
    return sorted(set(int(minutes) for minutes in POPULATION_ACCESS_CONTOURS_MIN))


def pedestrian_accessibility_minutes() -> list[int]:
    return sorted(set(int(minutes) for minutes in PEDESTRIAN_ACCESS_CONTOURS_MIN))


def fachgruppe_stock_columns_local() -> list[str]:
    return fachgruppe_stock_columns(FACHGRUPPE_IDS)


def own_cell_fachgruppe_columns() -> list[str]:
    return [f"own_cell_fachgruppe_{fachgruppe_id}_firms" for fachgruppe_id in FACHGRUPPE_IDS]


def own_cell_walk_fachgruppe_columns() -> list[str]:
    return [f"own_cell_walk_fachgruppe_{fachgruppe_id}_firms" for fachgruppe_id in FACHGRUPPE_IDS]


def accessibility_mass_source_columns() -> list[str]:
    return ["population_backcast", "active_firms_tminus1", *fachgruppe_stock_columns_local()]


def own_cell_mass_output_columns() -> tuple[str, ...]:
    return ("own_cell_pop", "own_cell_firms", *own_cell_fachgruppe_columns())


def own_cell_walk_mass_output_columns() -> tuple[str, ...]:
    return ("own_cell_walk_pop", "own_cell_walk_firms", *own_cell_walk_fachgruppe_columns())


def contour_mass_output_columns(minutes: int) -> tuple[str, ...]:
    return (
        f"pop_access_{minutes}min",
        f"existing_firms_access_{minutes}min",
        *[f"fachgruppe_{fachgruppe_id}_access_{minutes}min" for fachgruppe_id in FACHGRUPPE_IDS],
    )


def pedestrian_contour_mass_output_columns(minutes: int) -> tuple[str, ...]:
    return (
        f"walk_pop_{minutes}min",
        f"walk_firms_{minutes}min",
        *[f"walk_fachgruppe_{fachgruppe_id}_firms_{minutes}min" for fachgruppe_id in FACHGRUPPE_IDS],
    )


def pedestrian_pt_output_columns(minutes: int) -> tuple[str, ...]:
    return (f"walk_pt_stops_{minutes}min", f"walk_pt_departures_{minutes}min", f"walk_pt_routes_{minutes}min")


def routed_accessibility_output_columns() -> list[str]:
    columns = [column for minutes in accessibility_minutes() for column in (*contour_mass_output_columns(minutes), f"reachable_cells_{minutes}min")]
    for minutes in pedestrian_accessibility_minutes():
        columns.extend((*pedestrian_contour_mass_output_columns(minutes), *pedestrian_pt_output_columns(minutes), f"reachable_cells_walk_{minutes}min"))
    return columns


def integer_accessibility_output_columns() -> set[str]:
    columns = {f"reachable_cells_{minutes}min" for minutes in accessibility_minutes()}
    for minutes in pedestrian_accessibility_minutes():
        columns.update({f"walk_pt_stops_{minutes}min", f"walk_pt_routes_{minutes}min", f"reachable_cells_walk_{minutes}min"})
    columns.add("pt_ohne_haltestelle")
    return columns


def accessibility_output_columns() -> list[str]:
    columns = [*own_cell_mass_output_columns(), *own_cell_walk_mass_output_columns()]
    for minutes in accessibility_minutes():
        columns.extend(contour_mass_output_columns(minutes))
        columns.append(f"reachable_cells_{minutes}min")
    for minutes in pedestrian_accessibility_minutes():
        columns.extend(pedestrian_contour_mass_output_columns(minutes))
        columns.extend(pedestrian_pt_output_columns(minutes))
        columns.append(f"reachable_cells_walk_{minutes}min")
    columns.append("pt_ohne_haltestelle")
    return columns


def firm_accessibility_columns() -> list[str]:
    columns = ["own_cell_pop", "own_cell_firms", "own_cell_same_fachgruppe_firms", "own_cell_walk_pop", "own_cell_walk_firms", "own_cell_walk_same_fachgruppe_firms"]
    for minutes in accessibility_minutes():
        columns.extend([
            f"pop_access_{minutes}min",
            f"existing_firms_access_{minutes}min",
            f"same_fachgruppe_firms_access_{minutes}min",
        ])
    for minutes in pedestrian_accessibility_minutes():
        columns.extend([
            f"walk_pop_{minutes}min",
            f"walk_firms_{minutes}min",
            f"walk_same_fachgruppe_firms_{minutes}min",
        ])
    return columns


def quarter_table_for_year(year: int) -> pd.DataFrame:
    quarters = year_quarters(year)
    return pd.DataFrame({
        "year": quarters.year.astype(int),
        "quarter": quarters.quarter.astype(int),
        "period": quarters.astype(str),
    })


_raster_centroids_4326_cache: gpd.GeoDataFrame | None = None
_firm_accessibility_source_cache: pd.DataFrame | None = None


def load_yearly_accessibility_panel(year: int) -> pd.DataFrame:
    if not PANEL_PATH.exists():
        raise FileNotFoundError(f"Missing raster quarter panel file: {PANEL_PATH}")
    required_columns = [
        "grid_id",
        "year",
        "quarter",
        "period",
        "population_backcast",
        "active_firms_tminus1",
        *fachgruppe_stock_columns_local(),
    ]
    panel = pd.read_parquet(
        PANEL_PATH,
        columns=required_columns,
        filters=[("year", "==", year)],
    )
    numeric_columns = ["population_backcast", "active_firms_tminus1", *fachgruppe_stock_columns_local()]
    for column in numeric_columns:
        panel[column] = pd.to_numeric(panel[column], errors="coerce").fillna(0.0).astype(float)
    panel["year"] = pd.to_numeric(panel["year"], errors="coerce").astype(int)
    panel["quarter"] = pd.to_numeric(panel["quarter"], errors="coerce").astype(int)
    panel["period"] = panel["period"].astype(str)
    return panel


def load_raster_centroids_4326() -> gpd.GeoDataFrame:
    global _raster_centroids_4326_cache
    if _raster_centroids_4326_cache is not None:
        return _raster_centroids_4326_cache
    if not RASTER_PATH.exists():
        raise FileNotFoundError(f"Missing raster geometry file: {RASTER_PATH}")
    raster = gpd.read_parquet(RASTER_PATH, columns=["grid_id", "geometry"])
    raster["geometry"] = raster.geometry.centroid
    raster = raster.to_crs("EPSG:4326")[["grid_id", "geometry"]].set_index("grid_id")
    _raster_centroids_4326_cache = raster
    return _raster_centroids_4326_cache


def load_all_grid_cells_4326() -> gpd.GeoDataFrame:
    return load_raster_centroids_4326().reset_index().sort_values("grid_id", kind="stable").reset_index(drop=True)


def parse_route_ids(value) -> frozenset[str]:
    if pd.isna(value):
        return frozenset()
    return frozenset(item.strip() for item in str(value).split("|") if item.strip())


def load_pt_accessibility_destinations(year: int) -> gpd.GeoDataFrame:
    poi_path = POI_DIR / f"austria-{year}-pois.geoparquet"
    if not poi_path.exists():
        raise FileNotFoundError(f"Missing yearly POI file: {poi_path}")
    stops = gpd.read_parquet(poi_path)
    required = {"poi_type", "source_poi_id", "pt_departures_weekday", "pt_route_ids", "geometry"}
    missing = sorted(required - set(stops.columns))
    if missing:
        raise ValueError(f"{poi_path.name} is missing pedestrian PT fields: {missing}; rebuild yearly routing destinations")
    stops = stops.loc[stops["poi_type"].eq("pt_stop"), ["source_poi_id", "pt_departures_weekday", "pt_route_ids", "geometry"]].copy()
    stops = stops.dropna(subset=["source_poi_id", "geometry"])
    stops["parent_station_id"] = stops["source_poi_id"].astype(str)
    stops["pt_departures_weekday"] = pd.to_numeric(stops["pt_departures_weekday"], errors="coerce").fillna(0.0)
    stops["route_ids"] = stops["pt_route_ids"].map(parse_route_ids)
    if stops["parent_station_id"].duplicated().any():
        raise ValueError(f"{poi_path.name} contains duplicate parent PT station IDs")
    return stops.to_crs("EPSG:4326").sort_values("parent_station_id", kind="stable").reset_index(drop=True)


def load_accessibility_destinations_for_year(panel_year: pd.DataFrame) -> gpd.GeoDataFrame:
    mass_columns = accessibility_mass_source_columns()
    positive_grid_mask = panel_year.groupby("grid_id")[mass_columns].max().gt(0).any(axis=1)
    positive_grid_ids = positive_grid_mask.index[positive_grid_mask].tolist()
    if not positive_grid_ids:
        return gpd.GeoDataFrame(
            {"grid_id": pd.Series(dtype="object")},
            geometry=gpd.GeoSeries([], crs="EPSG:4326"),
            crs="EPSG:4326",
        )
    centroids = load_raster_centroids_4326()
    destinations = centroids.loc[centroids.index.intersection(positive_grid_ids)].reset_index()
    destinations = destinations.sort_values("grid_id", kind="stable").reset_index(drop=True)
    return gpd.GeoDataFrame(destinations, geometry="geometry", crs="EPSG:4326")


def accessibility_part_paths(parts_dir: Path) -> list[Path]:
    return sorted(parts_dir.glob("part_*.parquet"))


def accessibility_part_paths_match_schema(part_paths: list[Path]) -> bool:
    if not part_paths:
        return True
    required_columns = {"grid_id", "year", "quarter", "period", *accessibility_output_columns()}
    return all(required_columns.issubset(set(pq.ParquetFile(path).schema_arrow.names)) for path in part_paths)


def clear_accessibility_part_paths(part_paths: list[Path]) -> None:
    for part_path in part_paths:
        if part_path.exists():
            part_path.unlink()


def remove_accessibility_parts_dir(parts_dir: Path) -> None:
    part_paths = accessibility_part_paths(parts_dir)
    clear_accessibility_part_paths(part_paths)
    if parts_dir.exists() and not any(parts_dir.iterdir()):
        parts_dir.rmdir()


def completed_accessibility_origin_ids(part_paths: list[Path]) -> set:
    completed = set()
    for part_path in part_paths:
        part = pd.read_parquet(part_path, columns=["grid_id"])
        completed.update(part["grid_id"].dropna().unique())
    return completed


def accessibility_contour_minutes(feature: dict) -> int:
    properties = feature.get("properties", {})
    for key in ["contour", "time", "time_min", "time_minutes"]:
        value = properties.get(key)
        if finite_number(value):
            return int(round(float(value)))
        try:
            return int(round(float(value)))
        except (TypeError, ValueError):
            continue
    raise KeyError(f"Isochrone feature is missing a contour value in properties: {properties}")


def request_accessibility_isochrones(
    origin: tuple[str, float, float],
    costing: str = POPULATION_ACCESS_COSTING,
    contour_minutes: list[int] | None = None,
) -> dict[int, object]:
    grid_id, lat, lon = origin
    desired_contours = accessibility_minutes() if contour_minutes is None else sorted(set(contour_minutes))
    payload = {
        "locations": [{"lat": lat, "lon": lon}],
        "costing": costing,
        "contours": [{"time": minutes} for minutes in desired_contours],
        "polygons": POPULATION_ACCESS_POLYGONS,
        "denoise": POPULATION_ACCESS_DENOISE,
        "generalize": POPULATION_ACCESS_GENERALIZE_M,
    }
    last_error = None
    for attempt in range(1, POPULATION_ACCESS_REQUEST_RETRIES + 1):
        try:
            response = accessibility_http_session().post(
                f"{valhalla_endpoint()}/isochrone",
                json=payload,
                timeout=POPULATION_ACCESS_REQUEST_TIMEOUT_SECONDS,
            )
            if not response.ok:
                raise RuntimeError(
                    f"Valhalla isochrone request failed with HTTP {response.status_code}: {response.text[:1000]}"
                )
            data = response.json()
            features = data.get("features", [])
            if not features:
                raise RuntimeError(f"Unexpected Valhalla isochrone response keys: {sorted(data.keys())}")
            contour_geometries = {}
            for feature in features:
                contour_minutes = accessibility_contour_minutes(feature)
                if contour_minutes not in desired_contours:
                    continue
                geometry_payload = feature.get("geometry")
                if geometry_payload is None:
                    continue
                geometry = shape(geometry_payload)
                if geometry.is_empty:
                    continue
                if contour_minutes in contour_geometries:
                    contour_geometries[contour_minutes] = contour_geometries[contour_minutes].union(geometry)
                else:
                    contour_geometries[contour_minutes] = geometry
            missing = [minutes for minutes in desired_contours if minutes not in contour_geometries]
            if missing:
                raise RuntimeError(f"Isochrone response for {grid_id} is missing contours: {missing}")
            return contour_geometries
        except Exception as error:
            last_error = error
            if attempt == POPULATION_ACCESS_REQUEST_RETRIES:
                raise
            time.sleep(min(10, attempt * 2))
    raise RuntimeError(f"Isochrone request failed for {grid_id}: {last_error}")


def reachable_grid_positions_by_contour(
    origin: tuple[str, float, float],
    destination_cells: gpd.GeoDataFrame,
    destination_position_by_grid_id: dict[str, int],
    costing: str = POPULATION_ACCESS_COSTING,
    contour_minutes: list[int] | None = None,
) -> dict[int, np.ndarray]:
    minutes_to_query = accessibility_minutes() if contour_minutes is None else sorted(set(contour_minutes))
    contour_geometries = request_accessibility_isochrones(origin, costing, minutes_to_query)
    reachable = {}
    origin_position = destination_position_by_grid_id.get(origin[0])
    for minutes in minutes_to_query:
        contour_geometry = contour_geometries[minutes]
        positions = np.asarray(
            destination_cells.sindex.query(contour_geometry, predicate="intersects"),
            dtype=np.int64,
        )
        if origin_position is not None:
            positions = np.append(positions, origin_position)
        reachable[minutes] = np.unique(positions)
    return reachable


def positions_inside_contours(contour_geometries: dict[int, object], destinations: gpd.GeoDataFrame) -> dict[int, np.ndarray]:
    return {
        minutes: np.unique(np.asarray(destinations.sindex.query(geometry, predicate="intersects"), dtype=np.int64))
        for minutes, geometry in contour_geometries.items()
    }


def accessibility_records_for_origin(
    origin: tuple[str, float, float],
    year: int,
    quarterly_mass_arrays: dict[int, np.ndarray],
    quarter_period_map: dict[int, str],
    destination_cells: gpd.GeoDataFrame,
    destination_position_by_grid_id: dict[str, int],
    all_grid_cells: gpd.GeoDataFrame,
    all_grid_position_by_grid_id: dict[str, int],
    pt_destinations: gpd.GeoDataFrame,
    zero_mass_vector: np.ndarray,
    own_output_columns: tuple[str, ...],
    own_walk_output_columns: tuple[str, ...],
    contour_output_columns: dict[int, tuple[str, ...]],
    pedestrian_contour_output_columns: dict[int, tuple[str, ...]],
) -> list[dict]:
    grid_id = origin[0]
    skip_reason = ACCESSIBILITY_SKIPPED_ORIGINS.get(int(year), {}).get(str(grid_id))
    origin_position = destination_position_by_grid_id.get(grid_id)
    car_reachable = None
    car_grid_counts = None
    if skip_reason is None:
        car_contours = request_accessibility_isochrones(origin, POPULATION_ACCESS_COSTING, accessibility_minutes())
        car_reachable = positions_inside_contours(car_contours, destination_cells)
        if origin_position is not None:
            car_reachable = {minutes: np.unique(np.append(positions, origin_position)) for minutes, positions in car_reachable.items()}
        car_grid_positions = positions_inside_contours(car_contours, all_grid_cells)
        origin_all_grid_position = all_grid_position_by_grid_id.get(grid_id)
        if origin_all_grid_position is not None:
            car_grid_positions = {minutes: np.unique(np.append(positions, origin_all_grid_position)) for minutes, positions in car_grid_positions.items()}
        car_grid_counts = {minutes: len(positions) for minutes, positions in car_grid_positions.items()}

    walk_contours = request_accessibility_isochrones(origin, PEDESTRIAN_ACCESS_COSTING, pedestrian_accessibility_minutes())
    walk_reachable = positions_inside_contours(walk_contours, destination_cells)
    if origin_position is not None:
        walk_reachable = {minutes: np.unique(np.append(positions, origin_position)) for minutes, positions in walk_reachable.items()}
    walk_grid_positions = positions_inside_contours(walk_contours, all_grid_cells)
    origin_all_grid_position = all_grid_position_by_grid_id.get(grid_id)
    if origin_all_grid_position is not None:
        walk_grid_positions = {minutes: np.unique(np.append(positions, origin_all_grid_position)) for minutes, positions in walk_grid_positions.items()}
    walk_pt_positions = positions_inside_contours(walk_contours, pt_destinations)
    walk_pt_metrics = {}
    for minutes, positions in walk_pt_positions.items():
        reachable_stops = pt_destinations.iloc[positions]
        route_ids = set().union(*reachable_stops["route_ids"].tolist()) if len(reachable_stops) else set()
        walk_pt_metrics[minutes] = (
            int(reachable_stops["parent_station_id"].nunique()),
            float(reachable_stops["pt_departures_weekday"].sum()),
            int(len(route_ids)),
        )

    records = []
    for quarter, period in quarter_period_map.items():
        mass_matrix = quarterly_mass_arrays[quarter]
        record = {"grid_id": grid_id, "year": int(year), "quarter": int(quarter), "period": period}
        own_mass = zero_mass_vector if origin_position is None else mass_matrix[origin_position]
        record.update((column, float(value)) for column, value in zip(own_output_columns, own_mass, strict=True))
        record.update((column, float(value)) for column, value in zip(own_walk_output_columns, own_mass, strict=True))
        if skip_reason is not None:
            for minutes, columns in contour_output_columns.items():
                record.update((column, float("nan")) for column in columns)
                record[f"reachable_cells_{minutes}min"] = float("nan")
        else:
            for minutes, reachable_positions in car_reachable.items():
                accessible_mass = zero_mass_vector if reachable_positions.size == 0 else mass_matrix[reachable_positions].sum(axis=0, dtype=np.float64)
                record.update((column, float(value)) for column, value in zip(contour_output_columns[minutes], accessible_mass, strict=True))
                record[f"reachable_cells_{minutes}min"] = int(car_grid_counts[minutes])
        for minutes, reachable_positions in walk_reachable.items():
            accessible_mass = zero_mass_vector if reachable_positions.size == 0 else mass_matrix[reachable_positions].sum(axis=0, dtype=np.float64)
            record.update((column, float(value)) for column, value in zip(pedestrian_contour_output_columns[minutes], accessible_mass, strict=True))
            stop_count, departures, route_count = walk_pt_metrics[minutes]
            record[f"walk_pt_stops_{minutes}min"] = stop_count
            record[f"walk_pt_departures_{minutes}min"] = departures
            record[f"walk_pt_routes_{minutes}min"] = route_count
            record[f"reachable_cells_walk_{minutes}min"] = int(len(walk_grid_positions[minutes]))
        record["pt_ohne_haltestelle"] = int(record[f"walk_pt_stops_{max(pedestrian_accessibility_minutes())}min"] == 0)
        records.append(record)
    return records


def write_accessibility_potentials_output(
    year: int,
    part_paths: list[Path],
    output_path: Path,
    active_cells: pd.DataFrame,
) -> int:
    output = active_cells[["grid_id"]].copy().merge(quarter_table_for_year(year), how="cross")
    if part_paths:
        combined = pd.concat([pd.read_parquet(part_path) for part_path in part_paths], ignore_index=True)
        combined = combined.drop_duplicates(["grid_id", "year", "quarter"], keep="last")
        combined = combined.drop(columns=["period"], errors="ignore")
        output = output.merge(combined, on=["grid_id", "year", "quarter"], how="left")
    missing_columns = sorted(set(accessibility_output_columns()) - set(output.columns))
    if missing_columns:
        raise KeyError(f"Accessibility output is missing expected columns: {missing_columns}")
    for column in accessibility_output_columns():
        values = pd.to_numeric(output[column], errors="coerce")
        if column in integer_accessibility_output_columns():
            output[column] = values.astype("Int64")
        else:
            output[column] = values.astype(float) if column in routed_accessibility_output_columns() else values.fillna(0.0).astype(float)
    output["created_at"] = datetime.now().isoformat(timespec="seconds")
    output = output[["grid_id", "year", "quarter", "period", *accessibility_output_columns(), "created_at"]]
    tmp_path = output_path.with_name(output_path.name + ".tmp")
    output.to_parquet(tmp_path, index=False)
    tmp_path.replace(output_path)
    return len(output)


def prepare_accessibility_potential_inputs(year: int, active_cells: pd.DataFrame | None = None) -> dict:
    if active_cells is None:
        active_cells = load_active_cells_for_run()
    panel_year = load_yearly_accessibility_panel(year)
    destination_cells = load_accessibility_destinations_for_year(panel_year)
    _ = destination_cells.sindex
    all_grid_cells = load_all_grid_cells_4326()
    _ = all_grid_cells.sindex
    pt_destinations = load_pt_accessibility_destinations(year)
    _ = pt_destinations.sindex
    mass_source_columns = accessibility_mass_source_columns()
    destination_grid_ids = pd.Index(destination_cells["grid_id"])
    destination_position_by_grid_id = {
        grid_id: position for position, grid_id in enumerate(destination_grid_ids)
    }
    all_grid_position_by_grid_id = {grid_id: position for position, grid_id in enumerate(all_grid_cells["grid_id"])}
    quarter_period_map = {int(period.quarter): str(period) for period in year_quarters(year)}
    quarterly_mass_arrays = {}
    for quarter, period in quarter_period_map.items():
        quarter_frame = (
            panel_year.loc[panel_year["quarter"] == quarter, ["grid_id", *mass_source_columns]]
            .drop_duplicates("grid_id", keep="last")
            .set_index("grid_id")
            .reindex(destination_grid_ids, fill_value=0.0)
        )
        mass_matrix = np.ascontiguousarray(quarter_frame[mass_source_columns].to_numpy(dtype=np.float64, copy=True))
        mass_matrix.setflags(write=False)
        quarterly_mass_arrays[quarter] = mass_matrix
    zero_mass_vector = np.zeros(len(mass_source_columns), dtype=np.float64)
    zero_mass_vector.setflags(write=False)
    return {
        "active_cells": active_cells.reset_index(drop=True).copy(),
        "quarterly_mass_arrays": quarterly_mass_arrays,
        "quarter_period_map": quarter_period_map,
        "destination_cells": destination_cells,
        "destination_position_by_grid_id": destination_position_by_grid_id,
        "all_grid_cells": all_grid_cells,
        "all_grid_position_by_grid_id": all_grid_position_by_grid_id,
        "pt_destinations": pt_destinations,
        "zero_mass_vector": zero_mass_vector,
        "own_output_columns": own_cell_mass_output_columns(),
        "own_walk_output_columns": own_cell_walk_mass_output_columns(),
        "contour_output_columns": {
            minutes: contour_mass_output_columns(minutes) for minutes in accessibility_minutes()
        },
        "pedestrian_contour_output_columns": {
            minutes: pedestrian_contour_mass_output_columns(minutes) for minutes in pedestrian_accessibility_minutes()
        },
    }


def generate_accessibility_potentials(year: int, prepared_inputs: dict | None = None) -> Path:
    global LAST_FAILED_ORIGINS
    LAST_FAILED_ORIGINS = []
    paths = output_paths(year)
    if prepared_inputs is None:
        prepared_inputs = prepare_accessibility_potential_inputs(year)
    active_cells = prepared_inputs["active_cells"].copy()
    quarterly_mass_arrays = prepared_inputs["quarterly_mass_arrays"]
    quarter_period_map = prepared_inputs["quarter_period_map"]
    destination_cells = prepared_inputs["destination_cells"]
    destination_position_by_grid_id = prepared_inputs["destination_position_by_grid_id"]
    all_grid_cells = prepared_inputs["all_grid_cells"]
    all_grid_position_by_grid_id = prepared_inputs["all_grid_position_by_grid_id"]
    pt_destinations = prepared_inputs["pt_destinations"]
    zero_mass_vector = prepared_inputs["zero_mass_vector"]
    own_output_columns = prepared_inputs["own_output_columns"]
    own_walk_output_columns = prepared_inputs["own_walk_output_columns"]
    contour_output_columns = prepared_inputs["contour_output_columns"]
    pedestrian_contour_output_columns = prepared_inputs["pedestrian_contour_output_columns"]

    parts_dir = paths["potentials_parts"]
    parts_dir.mkdir(parents=True, exist_ok=True)
    part_paths = accessibility_part_paths(parts_dir)
    if part_paths and not accessibility_part_paths_match_schema(part_paths):
        print(
            "Detected legacy accessibility parts without quarter keys. "
            "Clearing incompatible part files and restarting the accessibility run for this year."
        )
        clear_accessibility_part_paths(part_paths)
        part_paths = []
    completed_origin_ids = completed_accessibility_origin_ids(part_paths)
    if completed_origin_ids:
        print(
            f"Resuming accessibility potentials from {len(part_paths):,} part files "
            f"with {len(completed_origin_ids):,} origins already represented"
        )

    records_buffer = []
    part_number = len(part_paths)

    def flush_records() -> None:
        nonlocal records_buffer, part_number
        if not records_buffer:
            return
        part_number += 1
        part_path = parts_dir / f"part_{part_number:05d}.parquet"
        part = pd.DataFrame(records_buffer)
        for column in accessibility_output_columns():
            values = pd.to_numeric(part[column], errors="coerce")
            if column in integer_accessibility_output_columns():
                part[column] = values.astype("Int64")
            else:
                part[column] = values.astype(float) if column in routed_accessibility_output_columns() else values.fillna(0.0).astype(float)
        part.to_parquet(part_path, index=False)
        part_paths.append(part_path)
        records_buffer = []
        gc.collect()

    def submit_next(executor, next_origin_index: int, futures: dict) -> int:
        while next_origin_index < len(active_cells):
            grid_id = active_cells["grid_id"].iloc[next_origin_index]
            if grid_id not in completed_origin_ids:
                origin = (
                    grid_id,
                    float(active_cells["lat"].iloc[next_origin_index]),
                    float(active_cells["lon"].iloc[next_origin_index]),
                )
                future = executor.submit(
                    accessibility_records_for_origin,
                    origin,
                    year,
                    quarterly_mass_arrays,
                    quarter_period_map,
                    destination_cells,
                    destination_position_by_grid_id,
                    all_grid_cells,
                    all_grid_position_by_grid_id,
                    pt_destinations,
                    zero_mass_vector,
                    own_output_columns,
                    own_walk_output_columns,
                    contour_output_columns,
                    pedestrian_contour_output_columns,
                )
                futures[future] = grid_id
                return next_origin_index + 1
            next_origin_index += 1
        return next_origin_index

    progress_bar = tqdm(
        total=len(active_cells),
        desc="Accessibility potentials",
        unit="origin",
        initial=len(completed_origin_ids),
        leave=True,
    )
    try:
        with ThreadPoolExecutor(max_workers=POPULATION_ACCESS_ORIGIN_WORKERS) as executor:
            futures = {}
            next_origin_index = 0
            while next_origin_index < len(active_cells) and len(futures) < POPULATION_ACCESS_MAX_IN_FLIGHT_ORIGINS:
                next_origin_index = submit_next(executor, next_origin_index, futures)

            while futures:
                for completed in as_completed(list(futures)):
                    grid_id = futures.pop(completed)
                    try:
                        origin_records = completed.result()
                    except Exception:
                        LAST_FAILED_ORIGINS.append(grid_id)
                        raise
                    if origin_records:
                        records_buffer.extend(origin_records)
                    completed_origin_ids.add(grid_id)
                    if len(records_buffer) >= POPULATION_ACCESS_FLUSH_ORIGINS * len(quarter_period_map):
                        flush_records()
                    progress_bar.update(1)
                    progress_bar.set_postfix_str(f"{len(completed_origin_ids):,} origins completed")
                    while next_origin_index < len(active_cells) and len(futures) < POPULATION_ACCESS_MAX_IN_FLIGHT_ORIGINS:
                        next_origin_index = submit_next(executor, next_origin_index, futures)
                    break
    finally:
        progress_bar.close()

    flush_records()
    if len(completed_origin_ids) != len(active_cells):
        raise RuntimeError(
            f"Accessibility potentials finished with {len(completed_origin_ids):,} completed origins, "
            f"expected {len(active_cells):,}"
        )
    total_rows = write_accessibility_potentials_output(year, part_paths, paths["potentials"], active_cells)
    print(f"Wrote {total_rows:,} rows to {paths['potentials']}")
    return paths["potentials"]


def load_firm_accessibility_source() -> pd.DataFrame:
    global _firm_accessibility_source_cache
    if _firm_accessibility_source_cache is not None:
        return _firm_accessibility_source_cache
    if not FIRMS_PATH.exists():
        raise FileNotFoundError(f"Missing firm assignment file: {FIRMS_PATH}")
    firms = pd.read_parquet(FIRMS_PATH, columns=["firm_id", "grid_id_100m", "Fachgruppe_ID", "founding_date", "exit_date"])
    firms = firms.dropna(subset=["firm_id", "grid_id_100m", "founding_date"]).copy()
    firms["founding_date"] = pd.to_datetime(firms["founding_date"], errors="coerce")
    firms["exit_date"] = pd.to_datetime(firms["exit_date"], errors="coerce")
    firms = firms.dropna(subset=["founding_date"]).copy()
    firms["exit_date_filled"] = firms["exit_date"].fillna(pd.Timestamp.max)
    firms["Fachgruppe_ID"] = firms["Fachgruppe_ID"].astype("string")
    firms["Fachgruppe_ID_normalized"] = firms["Fachgruppe_ID"]
    _firm_accessibility_source_cache = firms
    return _firm_accessibility_source_cache


def build_firm_quarter_panel_for_year(year: int) -> pd.DataFrame:
    firms = load_firm_accessibility_source()
    records = []
    for period in year_quarters(year):
        quarter_end = period.end_time.normalize()
        previous_quarter_end = (period - 1).end_time.normalize()
        active_in_period = firms[
            (firms["founding_date"] <= quarter_end)
            & (firms["exit_date_filled"] > previous_quarter_end)
        ].copy()
        if active_in_period.empty:
            continue
        active_in_period["year"] = int(period.year)
        active_in_period["quarter"] = int(period.quarter)
        active_in_period["period"] = str(period)
        active_in_period["included_in_lagged_stock"] = (
            (active_in_period["founding_date"] <= previous_quarter_end)
            & (active_in_period["exit_date_filled"] > previous_quarter_end)
        )
        records.append(
            active_in_period[[
                "firm_id",
                "grid_id_100m",
                "Fachgruppe_ID",
                "Fachgruppe_ID_normalized",
                "year",
                "quarter",
                "period",
                "included_in_lagged_stock",
            ]]
        )
    if not records:
        return pd.DataFrame(columns=[
            "firm_id",
            "grid_id_100m",
            "Fachgruppe_ID",
            "Fachgruppe_ID_normalized",
            "year",
            "quarter",
            "period",
            "included_in_lagged_stock",
        ])
    return pd.concat(records, ignore_index=True)


def generate_firm_accessibility_output(year: int, cell_accessibility_path: Path) -> Path:
    paths = output_paths(year)
    if not cell_accessibility_path.exists():
        raise FileNotFoundError(f"Missing cell accessibility output: {cell_accessibility_path}")
    cell_accessibility = pd.read_parquet(cell_accessibility_path).drop(columns=["created_at"], errors="ignore")
    cell_accessibility = cell_accessibility.rename(columns={"grid_id": "grid_id_100m"})
    firm_panel = build_firm_quarter_panel_for_year(year)
    joined = firm_panel.merge(
        cell_accessibility,
        on=["grid_id_100m", "year", "quarter", "period"],
        how="left",
    )
    fachgruppe_coverage = float(joined["Fachgruppe_ID_normalized"].isin(FACHGRUPPE_IDS).mean()) if len(joined) else 0.0
    if fachgruppe_coverage < 0.95:
        raise RuntimeError(f"Only {fachgruppe_coverage:.2%} of firm-quarter rows match configured Fachgruppe IDs; expected at least 95%.")
    missing_columns = sorted(set(accessibility_output_columns()) - set(joined.columns))
    if missing_columns:
        raise KeyError(f"Firm accessibility merge is missing expected columns: {missing_columns}")
    for column in accessibility_output_columns():
        values = pd.to_numeric(joined[column], errors="coerce")
        if column in integer_accessibility_output_columns():
            joined[column] = values.astype("Int64")
        else:
            joined[column] = values.astype(float) if column in routed_accessibility_output_columns() else values.fillna(0.0).astype(float)
    joined = joined.copy()  # Consolidate columns after the batch of dtype conversions.
    joined["own_cell_same_fachgruppe_firms"] = 0.0
    joined["own_cell_walk_same_fachgruppe_firms"] = 0.0
    for fachgruppe_id in FACHGRUPPE_IDS:
        mask = joined["Fachgruppe_ID_normalized"] == fachgruppe_id
        source_column = f"own_cell_fachgruppe_{fachgruppe_id}_firms"
        joined.loc[mask, "own_cell_same_fachgruppe_firms"] = joined.loc[mask, source_column]
        walk_source_column = f"own_cell_walk_fachgruppe_{fachgruppe_id}_firms"
        joined.loc[mask, "own_cell_walk_same_fachgruppe_firms"] = joined.loc[mask, walk_source_column]
    for minutes in accessibility_minutes():
        same_fachgruppe_column = f"same_fachgruppe_firms_access_{minutes}min"
        joined[same_fachgruppe_column] = 0.0
        for fachgruppe_id in FACHGRUPPE_IDS:
            source_column = f"fachgruppe_{fachgruppe_id}_access_{minutes}min"
            mask = joined["Fachgruppe_ID_normalized"] == fachgruppe_id
            joined.loc[mask, same_fachgruppe_column] = joined.loc[mask, source_column]
    for minutes in pedestrian_accessibility_minutes():
        same_fachgruppe_column = f"walk_same_fachgruppe_firms_{minutes}min"
        joined[same_fachgruppe_column] = 0.0
        for fachgruppe_id in FACHGRUPPE_IDS:
            source_column = f"walk_fachgruppe_{fachgruppe_id}_firms_{minutes}min"
            mask = joined["Fachgruppe_ID_normalized"] == fachgruppe_id
            joined.loc[mask, same_fachgruppe_column] = joined.loc[mask, source_column]
    zero_sum_columns = [
        f"same_fachgruppe_firms_access_{minutes}min"
        for minutes in accessibility_minutes()
        if float(joined[f"same_fachgruppe_firms_access_{minutes}min"].sum()) == 0.0
    ] + [
        f"walk_same_fachgruppe_firms_{minutes}min"
        for minutes in pedestrian_accessibility_minutes()
        if float(joined[f"walk_same_fachgruppe_firms_{minutes}min"].sum()) == 0.0
    ]
    if zero_sum_columns:
        raise RuntimeError(f"Same-Fachgruppe accessibility is zero for expected columns: {zero_sum_columns}")
    output_columns = [
        "firm_id",
        "grid_id_100m",
        "Fachgruppe_ID",
        "year",
        "quarter",
        "period",
        "included_in_lagged_stock",
        *firm_accessibility_columns(),
    ]
    output = joined[output_columns].copy()
    output["created_at"] = datetime.now().isoformat(timespec="seconds")
    tmp_path = paths["firm_accessibility"].with_name(paths["firm_accessibility"].name + ".tmp")
    output.to_parquet(tmp_path, index=False)
    tmp_path.replace(paths["firm_accessibility"])
    print(f"Wrote {len(output):,} firm-quarter rows to {paths['firm_accessibility']}")
    return paths["firm_accessibility"]


def split_fachgruppe_accessibility(wide_path: Path, pedestrian_path: Path, long_path: Path) -> None:
    """Stream the internal wide result into the two supported model products."""
    key_columns = ["grid_id", "year", "quarter"]
    main_columns = [*key_columns, "period", "own_cell_pop", "own_cell_firms", *main_access_columns(), *[f"reachable_cells_{minutes}min" for minutes in accessibility_minutes()], "created_at"]
    pedestrian_columns = [
        *key_columns, "period", "own_cell_walk_pop", "own_cell_walk_firms",
        *[column for minutes in pedestrian_accessibility_minutes() for column in (f"walk_pop_{minutes}min", f"walk_firms_{minutes}min", *pedestrian_pt_output_columns(minutes), f"reachable_cells_walk_{minutes}min")],
        "pt_ohne_haltestelle", "created_at",
    ]
    main_writer = None
    pedestrian_writer = None
    main_tmp = wide_path.with_name(wide_path.name + ".narrow.tmp")
    pedestrian_tmp = pedestrian_path.with_name(pedestrian_path.name + ".tmp")
    long_tmp = long_path.with_name(long_path.name + ".tmp")
    long_previous = long_path.with_name(long_path.name + ".previous")
    if main_tmp.exists():
        main_tmp.unlink()
    if pedestrian_tmp.exists():
        pedestrian_tmp.unlink()
    if long_tmp.exists():
        shutil.rmtree(long_tmp) if long_tmp.is_dir() else long_tmp.unlink()
    if long_previous.exists():
        shutil.rmtree(long_previous) if long_previous.is_dir() else long_previous.unlink()
    try:
        parquet = pq.ParquetFile(wide_path)
        projected_columns = list(dict.fromkeys([*main_columns, *pedestrian_columns]))
        for batch in parquet.iter_batches(batch_size=50_000, columns=projected_columns):
            frame = batch.to_pandas()
            main_table = pa.Table.from_pandas(frame[main_columns], preserve_index=False)
            pedestrian_table = pa.Table.from_pandas(frame[pedestrian_columns], preserve_index=False)
            if main_writer is None:
                main_writer = pq.ParquetWriter(main_tmp, main_table.schema, compression="snappy")
            main_writer.write_table(main_table)
            if pedestrian_writer is None:
                pedestrian_writer = pq.ParquetWriter(pedestrian_tmp, pedestrian_table.schema, compression="snappy")
            pedestrian_writer.write_table(pedestrian_table)
        parquet.close()
        parquet = None
        if main_writer:
            main_writer.close()
            main_writer = None
        if pedestrian_writer:
            pedestrian_writer.close()
            pedestrian_writer = None

        long_tmp.mkdir(parents=True, exist_ok=False)
        for fachgruppe_index, fachgruppe_id in enumerate(FACHGRUPPE_IDS):
            source_columns = [
                *key_columns,
                *[f"fachgruppe_{fachgruppe_id}_access_{minutes}min" for minutes in accessibility_minutes()],
                *[f"walk_fachgruppe_{fachgruppe_id}_firms_{minutes}min" for minutes in pedestrian_accessibility_minutes()],
                f"own_cell_fachgruppe_{fachgruppe_id}_firms",
                f"own_cell_walk_fachgruppe_{fachgruppe_id}_firms",
            ]
            fachgruppe_frames = []
            parquet = pq.ParquetFile(wide_path)
            for batch in parquet.iter_batches(batch_size=50_000, columns=source_columns):
                frame = batch.to_pandas()
                long_frame = frame[key_columns].copy()
                long_frame["Fachgruppe_ID"] = fachgruppe_id
                for minutes in accessibility_minutes():
                    long_frame[f"same_fachgruppe_firms_access_{minutes}min"] = frame[f"fachgruppe_{fachgruppe_id}_access_{minutes}min"].to_numpy()
                for minutes in pedestrian_accessibility_minutes():
                    long_frame[f"walk_same_fachgruppe_firms_{minutes}min"] = frame[f"walk_fachgruppe_{fachgruppe_id}_firms_{minutes}min"].to_numpy()
                long_frame["own_cell_same_fachgruppe_firms"] = frame[f"own_cell_fachgruppe_{fachgruppe_id}_firms"].to_numpy()
                long_frame["own_cell_walk_same_fachgruppe_firms"] = frame[f"own_cell_walk_fachgruppe_{fachgruppe_id}_firms"].to_numpy()
                fachgruppe_frames.append(long_frame)
            parquet.close()
            parquet = None
            long_table = pa.Table.from_pandas(pd.concat(fachgruppe_frames, ignore_index=True), preserve_index=False)
            pq.write_to_dataset(
                long_table,
                root_path=long_tmp,
                partition_cols=["Fachgruppe_ID"],
                basename_template=f"part-{fachgruppe_index:03d}-{{i}}.parquet",
                existing_data_behavior="overwrite_or_ignore",
                compression="snappy",
                max_rows_per_file=1_000_000,
                min_rows_per_group=128_000,
                row_group_size=128_000,
            )
            del fachgruppe_frames, long_table
            gc.collect()
    finally:
        if 'parquet' in locals() and parquet is not None: parquet.close()
        if main_writer: main_writer.close()
        if pedestrian_writer: pedestrian_writer.close()
    main_tmp.replace(wide_path)
    pedestrian_tmp.replace(pedestrian_path)
    if long_path.exists():
        long_path.replace(long_previous)
    try:
        long_tmp.replace(long_path)
    except Exception:
        if long_previous.exists() and not long_path.exists():
            long_previous.replace(long_path)
        raise
    if long_previous.exists():
        shutil.rmtree(long_previous) if long_previous.is_dir() else long_previous.unlink()

## Run

The single cell below runs every year sequentially. It resumes valid hidden checkpoints after interruption, restarts Valhalla after bounded failures, publishes only validated canonical files, records a per-year status row, and stops the service even after an error.

In [ ]:
if RUN_MODE not in {"dry-run", "smoke", "full"}:
    raise ValueError("RUN_MODE must be dry-run, smoke, or full")
for path in [ACTIVE_CELLS_PATH, PANEL_PATH, FIRMS_PATH, RASTER_PATH, *[POI_DIR / f"austria-{year}-pois.geoparquet" for year in YEARS_TO_RUN]]:
    if not path.exists(): raise FileNotFoundError(path)
if not EXTERNAL_VALHALLA:
    require_wsl_distribution()
    image_probe = run_wsl(f"docker image inspect {quote_bash(VALHALLA_IMAGE)} --format '{{{{.Id}}}}'", check=False)
    if image_probe.returncode != 0:
        raise RuntimeError(f"Pinned Valhalla image {VALHALLA_IMAGE!r} is not loaded locally; import/tag it before running. No network pull is attempted.")
    if image_probe.stdout.strip() != VALHALLA_IMAGE_ID:
        raise RuntimeError(f"Wrong Valhalla image ID: expected {VALHALLA_IMAGE_ID}, found {image_probe.stdout.strip()}")
pd.DataFrame({"setting": ["mode", "years", "Fachgruppen", "firm mass"], "value": [RUN_MODE, str(YEARS_TO_RUN), len(FACHGRUPPE_IDS), "active_firms_tminus1"]})

In [ ]:
WORK_ROOT = ROUTING_DATA / "work" / "routing_features"
WORK_ROOT.mkdir(parents=True, exist_ok=True)
ORIGINAL_OUTPUT_PATHS = output_paths
ORCHESTRATION_PHASE = "canonical"

def canonical_output_paths(year: int) -> dict:
    feature_dir = FEATURE_ROOT / str(year)
    feature_dir.mkdir(parents=True, exist_ok=True)
    return {
        "nearest": feature_dir / "nearest_infrastructure_100m.parquet",
        "nearest_partial": feature_dir / "nearest_infrastructure_100m.partial.parquet",
        "potentials": feature_dir / "accessibility_potentials_100m.parquet",
        "potentials_parts": feature_dir / ".accessibility_parts",
        "pedestrian_accessibility": feature_dir / "pedestrian_accessibility_quarter_100m.parquet",
        "fachgruppe_accessibility": feature_dir / "fachgruppe_accessibility_quarter_100m.parquet",
        "firm_accessibility": feature_dir / "firm_accessibility_quarter_100m.parquet",
    }

def output_paths(year: int) -> dict:
    canonical = canonical_output_paths(year)
    work_dir = WORK_ROOT / str(year)
    work_dir.mkdir(parents=True, exist_ok=True)
    slice_suffix = f".slice{ORIGIN_SLICE_INDEX}"
    return {
        "nearest": canonical["nearest"],
        "nearest_partial": work_dir / "nearest_infrastructure_100m.partial.parquet",
        "potentials": (work_dir / f"accessibility_potentials{slice_suffix}.parquet" if ORCHESTRATION_PHASE == "slice" else canonical["potentials"]),
        "potentials_parts": (work_dir / f".accessibility_parts{slice_suffix}" if ORCHESTRATION_PHASE == "slice" else canonical["potentials_parts"]),
        "pedestrian_accessibility": canonical["pedestrian_accessibility"],
        "fachgruppe_accessibility": canonical["fachgruppe_accessibility"],
        "firm_accessibility": (work_dir / "firm_accessibility.parquet" if ORCHESTRATION_PHASE == "merge" else canonical["firm_accessibility"]),
    }

def remove_known_visible_intermediates(year: int) -> None:
    feature_dir = FEATURE_ROOT / str(year)
    for pattern in ["*.partial", "*.partial.parquet", "*.slice*.parquet", "*.tmp", ".accessibility_parts*"]:
        for path in feature_dir.glob(pattern):
            if path.is_file(): path.unlink()
            elif path.is_dir(): shutil.rmtree(path)

def failed_manifest_path(year: int) -> Path:
    return WORK_ROOT / str(year) / "failed_origins.json"

def write_accessibility_skipped_origins_manifest(active_cells: pd.DataFrame) -> None:
    coordinates = active_cells.set_index("grid_id")[["lat", "lon"]]
    rows = []
    for skipped_year, skipped_origins in sorted(ACCESSIBILITY_SKIPPED_ORIGINS.items()):
        for grid_id, reason in sorted(skipped_origins.items()):
            if grid_id not in coordinates.index:
                raise KeyError(f"Configured skipped accessibility origin is absent from active cells: {grid_id}")
            coordinate = coordinates.loc[grid_id]
            rows.append({"year": int(skipped_year), "grid_id": grid_id, "lat": float(coordinate["lat"]), "lon": float(coordinate["lon"]), "reason": reason})
    ACCESSIBILITY_SKIPPED_ORIGINS_PATH.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(rows, columns=["year", "grid_id", "lat", "lon", "reason"]).to_csv(ACCESSIBILITY_SKIPPED_ORIGINS_PATH, index=False)

def write_failed_manifest(year: int, slice_index: int, attempt: int, error: Exception) -> None:
    path = failed_manifest_path(year)
    path.write_text(json.dumps({"year": year, "slice": slice_index, "attempt": attempt, "failed_origins": [str(origin) for origin in globals().get("LAST_FAILED_ORIGINS", [])], "error": repr(error), "recorded_at": datetime.now().isoformat(timespec="seconds")}, indent=2), encoding="utf-8")

def valid_slice_output(path: Path, expected_grid_ids: set[str], year: int) -> bool:
    if not path.exists(): return False
    try:
        required_columns = {"grid_id", "year", "quarter", "period", *accessibility_output_columns()}
        if not required_columns.issubset(set(pq.ParquetFile(path).schema_arrow.names)):
            return False
        frame = pd.read_parquet(path, columns=["grid_id", "year", "quarter"])
        return len(frame) == len(expected_grid_ids) * 4 and not frame.duplicated(["grid_id", "year", "quarter"]).any() and set(frame["grid_id"].astype(str)) == expected_grid_ids and set(frame["year"]) == {year} and set(frame["quarter"]) == {1, 2, 3, 4}
    except Exception:
        return False

def valid_nearest_output(path: Path, expected_grid_ids: set[str], year: int) -> bool:
    if not path.exists(): return False
    try:
        routed_types = DESTINATION_TYPES
        required_columns = {
            "grid_id", "year",
            *[f"{prefix}_{poi_type}{suffix}" for poi_type in routed_types for prefix, suffix in [("tt", "_min"), ("km", ""), ("nearest", "_id"), ("routing_status", "")]],
        }
        schema_columns = set(pq.ParquetFile(path).schema_arrow.names)
        legacy_pt_columns = {"tt_pt_stop_min", "km_pt_stop", "nearest_pt_stop_id", "routing_status_pt_stop", "has_pt_stop_5min_walk", "pt_departures_5min_walk"}
        if not required_columns.issubset(schema_columns) or legacy_pt_columns.intersection(schema_columns) or FORBIDDEN_NEAREST_COLUMNS.intersection(schema_columns):
            return False
        frame = pd.read_parquet(path, columns=["grid_id", "year"])
        return len(frame) == len(expected_grid_ids) and frame["grid_id"].is_unique and set(frame["grid_id"].astype(str)) == expected_grid_ids and set(frame["year"]) == {year}
    except Exception:
        return False

def run_accessibility_slice(year: int, slice_index: int, full_active_cells: pd.DataFrame) -> tuple[Path, int]:
    global ORIGIN_SLICE_INDEX, ORIGIN_SLICE_COUNT, ORCHESTRATION_PHASE
    ORIGIN_SLICE_INDEX = slice_index
    ORIGIN_SLICE_COUNT = AUTO_SLICE_COUNT
    ORCHESTRATION_PHASE = "slice"
    paths = output_paths(year)
    start = len(full_active_cells) * slice_index // AUTO_SLICE_COUNT
    stop = len(full_active_cells) * (slice_index + 1) // AUTO_SLICE_COUNT
    expected_ids = set(full_active_cells.iloc[start:stop]["grid_id"].astype(str))
    if valid_slice_output(paths["potentials"], expected_ids, year):
        return paths["potentials"], 0
    retries = 0
    for attempt in range(1, VALHALLA_RESTART_ATTEMPTS + 1):
        try:
            if not EXTERNAL_VALHALLA:
                start_valhalla_container(year)
            wait_until_valhalla_ready(year)
            result = generate_accessibility_potentials(year)
            if not valid_slice_output(result, expected_ids, year):
                raise RuntimeError(f"Slice {slice_index} for {year} failed row/key validation")
            return result, retries
        except Exception as error:
            retries += 1
            if not EXTERNAL_VALHALLA: stop_valhalla_container(year)
            if attempt == VALHALLA_RESTART_ATTEMPTS:
                write_failed_manifest(year, slice_index, attempt, error)
                raise RuntimeError(f"Retries exhausted for year {year}, slice {slice_index}; incomplete output was not published") from error
            time.sleep(min(60, 2 ** (attempt - 1)))
    raise AssertionError("unreachable")

def merge_accessibility_slices(year: int, full_active_cells: pd.DataFrame) -> dict:
    global ORCHESTRATION_PHASE
    work_dir = WORK_ROOT / str(year)
    slice_paths = [work_dir / f"accessibility_potentials.slice{index}.parquet" for index in range(AUTO_SLICE_COUNT)]
    if any(not path.exists() for path in slice_paths): raise FileNotFoundError(f"Missing accessibility slice for {year}: {[str(p) for p in slice_paths if not p.exists()]}")
    combined = pd.concat([pd.read_parquet(path) for path in slice_paths], ignore_index=True)
    expected_rows = len(full_active_cells) * 4
    if len(combined) != expected_rows or combined.duplicated(["grid_id", "year", "quarter"]).any() or set(combined["grid_id"].astype(str)) != set(full_active_cells["grid_id"].astype(str)):
        raise RuntimeError(f"Merged accessibility rows/keys invalid for {year}: {len(combined):,} rows, expected {expected_rows:,}")
    wide_tmp = work_dir / "accessibility_wide.parquet"
    combined.sort_values(["grid_id", "year", "quarter"]).to_parquet(wide_tmp, index=False)
    ORCHESTRATION_PHASE = "merge"
    canonical = canonical_output_paths(year)
    firm = generate_firm_accessibility_output(year, wide_tmp)
    split_fachgruppe_accessibility(wide_tmp, canonical["pedestrian_accessibility"], canonical["fachgruppe_accessibility"])
    firm_tmp = canonical["firm_accessibility"].with_name(canonical["firm_accessibility"].name + ".tmp")
    Path(firm).replace(firm_tmp)
    firm_tmp.replace(canonical["firm_accessibility"])
    published_tmp = canonical["potentials"].with_name(canonical["potentials"].name + ".tmp")
    wide_tmp.replace(published_tmp)
    published_tmp.replace(canonical["potentials"])
    return {"potentials": canonical["potentials"], "pedestrian": canonical["pedestrian_accessibility"], "firm": canonical["firm_accessibility"], "fachgruppe": canonical["fachgruppe_accessibility"], "rows": expected_rows}

def validate_published_year(year: int, full_active_cells: pd.DataFrame) -> dict:
    paths = canonical_output_paths(year)
    active_count = len(full_active_cells)
    nearest = pd.read_parquet(paths["nearest"])
    main = pd.read_parquet(paths["potentials"])
    pedestrian = pd.read_parquet(paths["pedestrian_accessibility"])
    firm = pd.read_parquet(paths["firm_accessibility"])
    if len(nearest) != active_count or not nearest["grid_id"].is_unique: raise RuntimeError(f"Nearest validation failed for {year}")
    if len(main) != active_count * 4 or main.duplicated(["grid_id", "year", "quarter"]).any() or set(main["year"]) != {year} or set(main["quarter"]) != {1,2,3,4}: raise RuntimeError(f"Grid-quarter validation failed for {year}")
    if len(pedestrian) != active_count * 4 or pedestrian.duplicated(["grid_id", "year", "quarter"]).any() or set(pedestrian["year"]) != {year} or set(pedestrian["quarter"]) != {1,2,3,4}: raise RuntimeError(f"Pedestrian grid-quarter validation failed for {year}")
    pedestrian_required = {"pt_ohne_haltestelle", *[column for minutes in pedestrian_accessibility_minutes() for column in (f"walk_pop_{minutes}min", f"walk_firms_{minutes}min", *pedestrian_pt_output_columns(minutes), f"reachable_cells_walk_{minutes}min")]}
    if not pedestrian_required.issubset(pedestrian.columns) or pedestrian[list(pedestrian_required)].isna().any().any(): raise RuntimeError(f"Pedestrian accessibility validation failed for {year}")
    routed_columns = main_access_columns()
    any_routed_missing = main[routed_columns].isna().any(axis=1)
    all_routed_missing = main[routed_columns].isna().all(axis=1)
    expected_skipped_ids = set(ACCESSIBILITY_SKIPPED_ORIGINS.get(int(year), {}))
    actual_skipped_ids = set(main.loc[any_routed_missing, "grid_id"].astype(str))
    if actual_skipped_ids != expected_skipped_ids or not (any_routed_missing == all_routed_missing).all() or int(any_routed_missing.sum()) != 4 * len(expected_skipped_ids):
        raise RuntimeError(f"Accessibility missing-value validation failed for {year}: expected skipped IDs={sorted(expected_skipped_ids)}, actual={sorted(actual_skipped_ids)}, missing rows={int(any_routed_missing.sum())}")
    long_files = sorted(paths["fachgruppe_accessibility"].rglob("*.parquet"))
    if not long_files: raise RuntimeError(f"Fachgruppe dataset is missing for {year}")
    long_glob = (paths["fachgruppe_accessibility"] / "**" / "*.parquet").as_posix()
    with duckdb.connect() as connection:
        long_rows, group_count, min_rows, max_rows, min_fachgruppen, max_fachgruppen, min_year, max_year = connection.execute(
            """
            WITH long_data AS (
                SELECT grid_id, year, quarter, CAST(Fachgruppe_ID AS VARCHAR) AS Fachgruppe_ID
                FROM read_parquet(?, hive_partitioning = true)
            ), coverage AS (
                SELECT grid_id, year, quarter, COUNT(*) AS row_count, COUNT(DISTINCT Fachgruppe_ID) AS fachgruppe_count
                FROM long_data GROUP BY grid_id, year, quarter
            )
            SELECT
                (SELECT COUNT(*) FROM long_data), COUNT(*), MIN(row_count), MAX(row_count),
                MIN(fachgruppe_count), MAX(fachgruppe_count),
                (SELECT MIN(year) FROM long_data), (SELECT MAX(year) FROM long_data)
            FROM coverage
            """,
            [long_glob],
        ).fetchone()
    expected_groups = active_count * 4
    expected_long_rows = expected_groups * len(FACHGRUPPE_IDS)
    if (long_rows, group_count, min_rows, max_rows, min_fachgruppen, max_fachgruppen, min_year, max_year) != (expected_long_rows, expected_groups, len(FACHGRUPPE_IDS), len(FACHGRUPPE_IDS), len(FACHGRUPPE_IDS), len(FACHGRUPPE_IDS), year, year):
        raise RuntimeError(f"Fachgruppe validation failed for {year}: rows={long_rows:,}, groups={group_count:,}, row coverage={min_rows}..{max_rows}, Fachgruppen={min_fachgruppen}..{max_fachgruppen}, years={min_year}..{max_year}")
    long_row_groups = sum(pq.ParquetFile(path).metadata.num_row_groups for path in long_files)
    maximum_row_groups = len(FACHGRUPPE_IDS) * math.ceil(expected_groups / 128_000)
    if long_row_groups > maximum_row_groups:
        raise RuntimeError(f"Fachgruppe dataset has {long_row_groups:,} row groups; expected at most {maximum_row_groups:,}")
    if len(firm) == 0 or set(pd.to_numeric(firm["year"], errors="coerce").dropna().astype(int)) != {year}: raise RuntimeError(f"Firm validation failed for {year}")
    return {"nearest_rows": len(nearest), "accessibility_rows": len(main), "pedestrian_rows": len(pedestrian), "firm_rows": len(firm), "fachgruppe_rows": long_rows, "fachgruppe_row_groups": long_row_groups}

def cleanup_successful_year(year: int) -> None:
    work_dir = WORK_ROOT / str(year)
    if work_dir.exists(): shutil.rmtree(work_dir)
    if REGENERATE_OUTPUTS: remove_known_visible_intermediates(year)

if RUN_MODE not in {"dry-run", "smoke", "full"}: raise ValueError("RUN_MODE must be dry-run, smoke, or full")
if ROUTING_STATUS_PATH.exists():
    previous_status = pd.read_csv(ROUTING_STATUS_PATH)
    status_rows = previous_status.loc[~pd.to_numeric(previous_status["year"], errors="coerce").isin(YEARS_TO_RUN)].to_dict(orient="records")
else:
    status_rows = []
write_accessibility_skipped_origins_manifest(pd.read_parquet(ACTIVE_CELLS_PATH, columns=["grid_id", "lat", "lon"]))
for year in YEARS_TO_RUN:
    started_at = datetime.now()
    full_active = pd.read_parquet(ACTIVE_CELLS_PATH)
    if MAX_ACTIVE_CELLS is not None: full_active = full_active.head(MAX_ACTIVE_CELLS).copy()
    remove_known_visible_intermediates(year)
    if RUN_MODE == "dry-run":
        status_rows.append({"year": year, "status": "dry-run", "start_time": started_at.isoformat(timespec="seconds")})
        continue
    try:
        if not EXTERNAL_VALHALLA: start_valhalla_container(year)
        wait_until_valhalla_ready(year)
        if RUN_MODE == "smoke":
            print({"year": year, "graz_route": valhalla_test_route(), "writes": 0})
            status_rows.append({"year": year, "status": "smoke", "start_time": started_at.isoformat(timespec="seconds")})
            continue
        nearest_retries = 0
        if RUN_NEAREST_INFRASTRUCTURE:
            ORCHESTRATION_PHASE = "nearest"
            canonical_nearest = canonical_output_paths(year)["nearest"]
            expected_grid_ids = set(full_active["grid_id"].astype(str))
            if valid_nearest_output(canonical_nearest, expected_grid_ids, year):
                print(f"Resume {year}: published nearest-infrastructure output is already valid; skipping routing")
            else:
                for nearest_attempt in range(1, VALHALLA_RESTART_ATTEMPTS + 1):
                    try:
                        generate_nearest_infrastructure(year)
                        break
                    except Exception as error:
                        nearest_retries += 1
                        if not EXTERNAL_VALHALLA: stop_valhalla_container(year)
                        if nearest_attempt == VALHALLA_RESTART_ATTEMPTS: raise RuntimeError(f"Retries exhausted for nearest infrastructure {year}") from error
                        if not EXTERNAL_VALHALLA: start_valhalla_container(year)
                        wait_until_valhalla_ready(year)
        slice_retries = 0
        for slice_index in range(AUTO_SLICE_COUNT):
            _, retries = run_accessibility_slice(year, slice_index, full_active)
            slice_retries += retries
        merged = merge_accessibility_slices(year, full_active)
        counts = validate_published_year(year, full_active)
        cleanup_successful_year(year)
        finished_at = datetime.now()
        status_rows.append({"year": year, "status": "done", "start_time": started_at.isoformat(timespec="seconds"), "end_time": finished_at.isoformat(timespec="seconds"), "duration_minutes": round((finished_at-started_at).total_seconds()/60, 2), "slice_count": AUTO_SLICE_COUNT, "retries": slice_retries + nearest_retries, "failures": 0, "nearest_rows": counts["nearest_rows"], "accessibility_rows": counts["accessibility_rows"], "pedestrian_rows": counts["pedestrian_rows"], "firm_rows": counts["firm_rows"], "fachgruppe_rows": counts["fachgruppe_rows"], "firm_mass_source": "active_firms_tminus1", "output_paths": json.dumps({k: str(v) for k,v in merged.items() if k != "rows"})})
    except Exception as error:
        finished_at = datetime.now()
        status_rows.append({"year": year, "status": "failed", "start_time": started_at.isoformat(timespec="seconds"), "end_time": finished_at.isoformat(timespec="seconds"), "duration_minutes": round((finished_at-started_at).total_seconds()/60, 2), "error": repr(error), "firm_mass_source": "active_firms_tminus1"})
        pd.DataFrame(status_rows).to_csv(ROUTING_STATUS_PATH, index=False)
        raise
    finally:
        if not EXTERNAL_VALHALLA: stop_valhalla_container(year)
    pd.DataFrame(status_rows).to_csv(ROUTING_STATUS_PATH, index=False)
pd.DataFrame(status_rows)

## Public-transport walking features


In [ ]:
pt_feature_columns = [
    "walk_pt_stops_10min",
    "walk_pt_departures_10min",
    "walk_pt_routes_10min",
    "pt_ohne_haltestelle",
]
pt_feature_frames = []
for year in YEARS_TO_RUN:
    path = FEATURE_ROOT / str(year) / "pedestrian_accessibility_quarter_100m.parquet"
    if path.exists():
        frame = pd.read_parquet(path, columns=pt_feature_columns)
        frame["year"] = year
        pt_feature_frames.append(frame)

if pt_feature_frames:
    pt_features = pd.concat(pt_feature_frames, ignore_index=True)
    display(pt_features[pt_feature_columns].describe().round(2))
    display(pt_features.groupby("year")[pt_feature_columns].mean().round(2))
